### Froze Lake using Q-learning

In [1]:
# Start with importing the library 
import numpy as np
import gym 
import os 
import time 

In [4]:
# Define the Q-learning agent
class QLearningAgent:
    def __init__(self, env, learning_rate=0.8, discount_factor=0.95, epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01):
        """
        Initialize the Q-learning agent.
        """
        self.env = env
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        # Initialize Q-table with zeros for all state-action pairs
        self.q_table = np.zeros((env.observation_space.n, env.action_space.n))

    def choose_action(self, state):
        """
        Choose an action using an ε-greedy policy.
        """
        if np.random.uniform(0, 1) < self.epsilon:  # Exploration
            return self.env.action_space.sample()
        else:  # Exploitation
            return np.argmax(self.q_table[state, :])

    def update_q_value(self, state, action, reward, next_state):
        """
        Update the Q-value for the given state-action pair.
        """
        best_next_action = np.max(self.q_table[next_state, :])  # Get max Q-value for next state
        td_target = reward + self.discount_factor * best_next_action  # Compute TD target
        td_error = td_target - self.q_table[state, action]  # TD error
        self.q_table[state, action] += self.learning_rate * td_error  # Update Q-value

    def decay_epsilon(self):
        """
        Decay the exploration rate ε after each episode.
        """
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

Next we defien train loop.

In [5]:
def train(agent, env, episodes=2000, max_steps_per_episode=100, video_dir="./videos"):
    """
    Train the Q-learning agent with optional video recording
    """
    # Create a directory for video storage if not exists 
    if not os.path.exists(video_dir):
        os.makedirs(video_dir)

    # Record the environment during training (every 100 episodes)
    env = gym.wrappers.RecordVideo(env, video_dir, episode_trigger=lambda e: e % 100 == 0)

    for episode in range(episodes):
        state = env.reset()  # Reset environment at the start of each episode
        done = False
        steps = 0

        while not done and steps < max_steps_per_episode:
            # Choose action
            action = agent.choose_action(state)

            # Take action and get the next state, reward, and done flag
            next_state, reward, done, _ = env.step(action)

            # Update the Q-table
            agent.update_q_value(state, action, reward, next_state)

            # Move to the next state
            state = next_state
            steps += 1

        # Decay exploration rate after each episode
        agent.decay_epsilon()

        # Print progress every 100 episodes
        if (episode + 1) % 100 == 0:
            print(f"Episode {episode + 1}/{episodes} - Epsilon: {agent.epsilon:.4f}")

    print("Training completed. Videos saved in:", video_dir)


In [6]:
def test(agent, env, episodes=5, max_steps_per_episode=100, video_dir="./videos/test"):
    """
    Test the trained Q-learning agent with text-based rendering and optional video recording.
    """
    # Create a directory for video storage if not exists
    if not os.path.exists(video_dir):
        os.makedirs(video_dir)

    # Record the environment during testing
    env = gym.wrappers.RecordVideo(env, video_dir, episode_trigger=lambda e: True)

    for episode in range(episodes):
        state = env.reset()
        done = False
        steps = 0
        print(f"\nTest Episode {episode + 1}")

        while not done and steps < max_steps_per_episode:
            print(env.render(mode="ansi"))  # Use text-based rendering in the console
            action = np.argmax(agent.q_table[state, :])  # Choose action with the highest Q-value
            state, reward, done, _ = env.step(action)
            steps += 1

            if done:
                if reward == 1:
                    print("Reached the goal!")
                else:
                    print("Fell into a hole.")

    env.close()  # Close environment after testing
    print("Testing completed. Videos saved in:", video_dir)


In [7]:
if __name__ == "__main__":
    # Create FrozenLake environment without render_mode for older Gym versions
    env = gym.make("FrozenLake-v1")  # No render_mode

    # Initialize Q-learning agent
    agent = QLearningAgent(env)

    # Train the agent
    train(agent, env)

    # Test the trained agent and save video of performance
    test(agent, env, video_dir="./videos")

C:\Users\utkri\anaconda3\envs\pytorch_env\Lib\site-packages\gym\wrappers\record_video.py:41: UserWarning: WARN: Overwriting existing videos at C:\Users\utkri\RL_Programs\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode 100/2000 - Epsilon: 0.6058
Episode 200/2000 - Epsilon: 0.3670
Episode 300/2000 - Epsilon: 0.2223
Episode 400/2000 - Epsilon: 0.1347
Episode 500/2000 - Epsilon: 0.0816
Episode 600/2000 - Epsilon: 0.0494
Episode 700/2000 - Epsilon: 0.0299
Episode 800/2000 - Epsilon: 0.0181
Episode 900/2000 - Epsilon: 0.0110
Episode 1000/2000 - Epsilon: 0.0100
Episode 1100/2000 - Epsilon: 0.0100
Episode 1200/2000 - Epsilon: 0.0100
Episode 1300/2000 - Epsilon: 0.0100
Episode 1400/2000 - Epsilon: 0.0100
Episode 1500/2000 - Epsilon: 0.0100
Episode 1600/2000 - Epsilon: 0.0100
Episode 1700/2000 - Epsilon: 0.0100
Episode 1800/2000 - Epsilon: 0.0100
Episode 1900/2000 - Epsilon: 0.0100
Episode 2000/2000 - Epsilon: 0.0100
Training completed. Videos saved in: ./videos

Test Episode 1

SFFF
FHFH
FFFH
HFFG

  (Left)
SFFF
FHFH
FFFH
HFFG

  (Left)
SFFF
FHFH
FFFH
HFFG

  (Left)
SFFF
FHFH
FFFH
HFFG

  (Up)
SFFF
FHFH
FFFH
HFFG

  (Down)
SFFF
FHFH
FFFH
HFFG

  (Right)
SFFF
FHFH
FFFH
HFFG

Reached the goal!

Test Ep